In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Cargar archivos csv

In [3]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [4]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
                                      if pd.notna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

    # eliminar , al final del ejercicio
    df.ejercicio = df.ejercicio.str.strip(",")

    return df

Arreglar ejercicios de Ponencia

In [5]:
def fix_ponencia(row):
    if "PONENCIA" in row["info_ejercicio"]:
        row["convocatoria"] = "Ponencia"
        pos_ejercicio = row["info_ejercicio"].find("EJERCICIO")
        row["ejercicio"] = row["info_ejercicio"][pos_ejercicio:].title()
    return row

In [6]:
df = df.apply(fix_ponencia, axis = 1)

In [7]:
df[df.ejercicio.isna()]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2556,Mates II,Mates II,2017,NaN,NaN,Matrices Y Determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,enunciados_ejercicios/Mates II/matrices_determ...,enunciados_ejercicios/Mates II/matrices_determ...,NaN


In [8]:
print(df[df.ejercicio.isna()].info_ejercicio)

2556    MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...
Name: info_ejercicio, dtype: str


In [9]:
df.loc[2556, ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [10]:
df.sample(3)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
3215,Química,Química,2003,Reserva 2,Ejercicio 5. Opción A,Cantidad Química,QUÍMICA. 2003. RESERVA 2. EJERCICIO 5. OPCIÓN A,Al tratar 5 g de galena con ácido sulfúrico se...,enunciados_ejercicios/Química/cantidad_quími...,enunciados_ejercicios/Química/cantidad_quími...,NaN
913,Mates CCSS,Mates CCSS,2009,Septiembre,Ejercicio 3 Parte Ii Opción B,Inferencia Estadística,SOCIALES II. 2009 SEPTIEMBRE. EJERCICIO 3 PART...,El tiempo que se tarda en la caja de un superm...,enunciados_ejercicios/Mates CCSS/inferencia_es...,enunciados_ejercicios/Mates CCSS/inferencia_es...,NaN
1863,Mates II,Mates II,2003,Reserva 3,Ejercicio 2. Opción A.,Integrales,MATEMÁTICAS II. 2003. RESERVA 3. EJERCICIO 2. ...,Se sabe que la función f :  definida por f(x)...,enunciados_ejercicios/Mates II/integrales/Mate...,enunciados_ejercicios/Mates II/integrales/png/...,NaN


Limpiar los enunciados

In [11]:
df = clean_dataframe(df)

Tras la limpieza completa, creo el ID

In [12]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    # se usa str() para convertir año (int) en string
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [13]:
# create id
df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
# rearrange columns
df = df[["id"] + list(df.columns[:-2])]

Exportar el dataframe limpio

In [14]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [15]:
df.sample(5)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
3520,quimica_conf_electronica_2003_reserva_4_ejerci...,Química,Química,2003,Reserva 4,"Ejercicio 3, opción B",Conf. Electrónica,QUÍMICA. 2003. RESERVA 4. EJERCICIO 3. OPCIÓN B,a) Escriba las configuraciones electrónicas de...,enunciados_ejercicios/Química/conf._electrón...,enunciados_ejercicios/Química/conf._electrón...
902,mates_ccss_inferencia_estadistica_2018_reserva...,Mates CCSS,Mates CCSS,2018,Reserva 4,Ejercicio 4 opción A,Inferencia Estadística,SOCIALES II. 2018 RESERVA 4. EJERCICIO 4 OPCIÓN A,La calificación que obtiene el alumnado en una...,enunciados_ejercicios/Mates CCSS/inferencia_es...,enunciados_ejercicios/Mates CCSS/inferencia_es...
265,fisica_fisica_cuantica_y_nuclear_2024_reserva_...,Física,Física,2024,Reserva 2,Ejercicio D1,Física cuántica y nuclear,FISICA. 2024. RESERVA 2. EJERCICIO D1,a) i) Describa las características de las emis...,enunciados_ejercicios/Física/física_cuántic...,enunciados_ejercicios/Física/física_cuántic...
4357,quimica_reacciones_redox_2008_reserva_4_ejerci...,Química,Química,2008,Reserva 4,"Ejercicio 6, opción B",Reacciones Redox,QUÍMICA. 2008. RESERVA 4. EJERCICIO 6. OPCIÓN B,a) Calcule el tiempo necesario para que una co...,enunciados_ejercicios/Química/redox/Química_...,enunciados_ejercicios/Química/redox/png/png_Q...
3742,quimica_equilibrio_quimico_2018_reserva_2_ejer...,Química,Química,2018,Reserva 2,"Ejercicio 5, opción B",Equilibrio Químico,QUÍMICA. 2018. RESERVA 2. EJERCICIO 5. OPCIÓN B,"En un recipiente de 2 L se introducen 4,90 g d...",enunciados_ejercicios/Química/equilibrio_quí...,enunciados_ejercicios/Química/equilibrio_quí...


In [16]:
df.isna().sum()

id                    0
asignatura            0
asignatura_carpeta    0
año                   0
convocatoria          0
ejercicio             0
tema                  0
info_ejercicio        0
enunciado             4
archivo               0
imagen                0
dtype: int64

In [17]:
df[df.enunciado.isna()]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
442,fisica_ondas_2016_reserva_3_ejercicio_4_opcion_a,Física,Física,2016,Reserva 3,"Ejercicio 4, opción A",Ondas,FISICA. 2016. RESERVA 3. EJERCICIO 4. OPCIÓN A,NaN,enunciados_ejercicios/Física/movimiento_ondul...,enunciados_ejercicios/Física/movimiento_ondul...
4204,quimica_reacciones_redox_2026_junio_ejercicio_5,Química,Química,2026,Junio,Ejercicio 5,Reacciones Redox,QUÍMICA. 2026. JUNIO. EJERCICIO 5,NaN,enunciados_ejercicios/Química/redox/Química_...,enunciados_ejercicios/Química/redox/png/png_Q...
4652,quimica_termoquimica_2025_julio_ejercicio_5,Química,Química,2025,Julio,Ejercicio 5,Termoquímica,QUÍMICA.2025. JULIO. EJERCICIO 5,NaN,enunciados_ejercicios/Química/termoquímica/Q...,enunciados_ejercicios/Química/termoquímica/p...
4655,quimica_termoquimica_2025_reserva_2_ejercicio_5,Química,Química,2025,Reserva 2,Ejercicio 5,Termoquímica,QUÍMICA.2025. RESERVA 2. EJERCICIO 5,NaN,enunciados_ejercicios/Química/termoquímica/Q...,enunciados_ejercicios/Química/termoquímica/p...


In [18]:
df.asignatura.unique()

<ArrowStringArray>
['Física', 'Mates CCSS', 'Mates II', 'Química']
Length: 4, dtype: str

In [19]:
df.loc[:, ["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Física,Campo eléctrico y magnético
121,Física,Campo gravitatorio
244,Física,Física cuántica y nuclear
362,Física,Ondas
449,Física,Óptica geométrica
486,Mates CCSS,Contraste De Hipótesis
545,Mates CCSS,Distribución Binomial
546,Mates CCSS,Distribución Binomial
550,Mates CCSS,Funciones CCSS
806,Mates CCSS,Inferencia Estadística


Arreglar "Distribución binomial "

In [20]:
df.tema = df.tema.str.strip()

In [21]:
df[df.asignatura == "Mates CCSS"].tema.unique()

<ArrowStringArray>
[        'Contraste De Hipótesis',          'Distribución Binomial',
                  'Funciones CCSS',         'Inferencia Estadística',
        'Matrices Y Determinantes',                    'Probabilidad',
            'Programación Lineal', 'Sistemas de Ecuaciones Lineales']
Length: 8, dtype: str

In [22]:
df.convocatoria.unique()

<ArrowStringArray>
[             'Julio',              'Junio',          'Reserva 1',
          'Reserva 2',          'Reserva 3',          'Reserva 4',
         'Septiembre', 'Reserva1 Ejercicio',           'Ponencia',
             'Modelo']
Length: 10, dtype: str

In [23]:
df.loc[df.convocatoria == "Reserva1 Ejercicio",
       ["convocatoria", "ejercicio"]] = ("Reserva 1", "Ejercicio 1")

In [24]:
df.to_csv(root/"csv_enunciados/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")

In [25]:
df = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [26]:
df.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
2182,mates_ii_geometria_2012_reserva_2_ejercicio_4_...,Mates II,Mates II,2012,Reserva 2,"Ejercicio 4, opción B",Geometría,MATEMÁTICAS II. 2012. RESERVA 2. EJERCICIO 4. ...,"L os puntos A(1,1,5) y B(1,1,2) son vértices c...",enunciados_ejercicios/Mates II/geometría/Mate...,enunciados_ejercicios/Mates II/geometría/png/...
3401,quimica_conf_electronica_2020_junio_b1,Química,Química,2020,Junio,B1,Conf. Electrónica,QUÍMICA. 2020. JUNIO B1,Dado un elemento de número atómico 20:\na) Esc...,enunciados_ejercicios/Química/conf._electrón...,enunciados_ejercicios/Química/conf._electrón...
2862,mates_ii_sistemas_de_ecuaciones_lineales_2016_...,Mates II,Mates II,2016,Junio,"Ejercicio 3, opción B",Sistemas De Ecuaciones Lineales,MATEMÁTICAS II. 2016. JUNIO. EJERCICIO 3. OPCI...,(31)x2y 5\n\nSe considera el sistema d...,enunciados_ejercicios/Mates II/sistemas_de_ecu...,enunciados_ejercicios/Mates II/sistemas_de_ecu...
